# Python, part 5 · Your Own Session 2

**Goal:** put your functions from parts 1 to 3 together into your own version of course 1's session 2, prove it gives course 1's answers, and see the same steps in course 2's SQL.

Part 0 opened the one line `cleaning.session2(df)` and found six named steps. Today you write the steps yourself.

**How to use this notebook** (the same as course 1)
- Start the lab cluster first (**Compute**; a cold start takes 5 minutes or more), then attach this notebook to it.
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = "P5"

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import check, cleaning, inside, pipeline
from lnl.check import ___

## 1. Three questions, asked of the file as written
Each of these went wrong somewhere in parts 1 to 3. Here they are again, asked of the raw file with no cleaning at all:

In [ ]:
raw = pipeline.load_raw()

print("Records missing a custody level (Q1):         ", (raw["custody_level"] == "").sum())
print("Records with no disciplinary points recorded:", (raw["disciplinary_points"] == "").sum())
print("Records named KENT (Q3):                       ", (raw["last_name"] == "KENT").sum())

Every one is wrong: spaces hide some blanks, `N/A` hides more, and capitals hide the KENTs. By the end of this notebook, your own code will get all three right.

## 2. Your functions so far
Here are the functions you wrote, with the answers filled in so everyone starts from the same place:
- `match_key` from part 1;
- `clean_value` from part 2;
- `my_strip_and_null` from part 3.

`tokens` is course 1's data dictionary, the same as in part 2.

In [ ]:
def match_key(name):
    """The same name, however it was typed: no spaces at the ends, all capitals."""
    return name.strip().upper()


def clean_value(value, codes):
    """None (Python's empty value) when value is one of the codes; otherwise value, as it was."""
    if value in codes:
        return None
    return value


def my_strip_and_null(df):
    """Trim spaces and tabs from both ends of every value, then turn empty text into a real empty value."""
    out = df.copy()
    for col in out.columns:
        stripped = out[col].str.strip()
        out[col] = stripped.mask(stripped == "")
    return out


tokens = cleaning.NULL_TOKENS

## 3. One more function: `clean_value` for a whole column
`clean_value` works on one value. To use it on a whole column you need a loop, and the short way to write one is a **list made by a loop on one line**. You read one in part 2, inside course 1's `apply_null_tokens`:

```python
[clean_value(v, codes) for v in out[column]]
```

Read it right to left: *for each value `v` in the column, call `clean_value(v, codes)`, and collect the results in a list.*

🧪 **Your turn #1.** Finish `null_codes`. For every column in the data dictionary, it replaces the column with your cleaned values. The check runs it after `my_strip_and_null` and compares the result with course 1's `apply_null_tokens`, cell for cell.

In [ ]:
def null_codes(df):
    """Every data-dictionary code made empty, column by column, with part 2's clean_value."""
    out = df.copy()
    for column, codes in tokens.items():
        cleaned = ___      # clean_value for every value in out[column], as a list
        out[column] = cleaned
    return out

check.p5_ex1(null_codes)

## 4. Your own session 2
Now chain the steps, the way part 0 showed `session2` does: each step takes the table and hands back a new one.

Step 3 adds the match key. `.dropna()` leaves out the empty names, because `match_key` can't strip an empty value. pandas then puts each result back on its own row, so the rows that were left out stay empty.

🧪 **Your turn #2.** Fill in step 1. The check runs your whole pipeline on all 100,000 records and compares every cell with course 1.

In [ ]:
def my_session2(df):
    """Course 1's session 2, the parts you wrote: strip and null, the don't-know codes, a match key for last names."""
    df = ___                                                       # step 1: part 3's function, on df
    df = null_codes(df)                                            # step 2: part 2's clean_value, every coded column
    df["last_name_std"] = df["last_name"].dropna().map(match_key)  # step 3: part 1's match_key
    return df

check.p5_ex2(my_session2)

## 5. The payoff: section 1's questions, from your own table

In [ ]:
check.p5_payoff(my_session2)

All three questions now get course 1's answers, from code you wrote. Course 1's own session 2 does more than this (it repairs damaged text, folds accents, proves and converts the IDs, maps the coded values), but every one of those steps is built from the same pieces you now know.

## 6. The same steps in course 2
Course 2 builds a lakehouse, and its cleaning is written in **SQL** rather than Python. Here is how course 2 cleans a text cell, quoted from its code (`dbt/macros/world.sql`, as of 2026-11-05). `{{ x }}` stands for the column being cleaned:

```sql
-- Tokens that stand for "no value"; blanks and tabs count too.
(({{ x }} <> '' AND trim({{ x }}, ' ' || chr(9)) = '')
 OR upper(trim({{ x }}, ' ' || chr(9))) IN ('UNK', 'UNKNOWN', 'N/A', '?', '00/00/0000', 'NULL', 'NONE'))

-- A cell as text: trimmed, with tokens and blanks as null.
CASE WHEN <the test above> THEN NULL ELSE nullif(trim({{ x }}, ' ' || chr(9)), '') END
```

You can read it with what you know:

| SQL | Python you wrote |
| --- | --- |
| `trim(x, ' ' \|\| chr(9))` (`chr(9)` is a tab) | `.strip()` |
| `nullif(..., '')` | `.mask(stripped == "")` |
| `upper(...) IN ('UNK', 'N/A', ...)` | `value in codes`, after `.upper()` |
| `CASE WHEN ... THEN NULL ELSE ... END` | `if unknown: return None` ... `return value` |

Course 2's list of codes is longer, because its data comes from other source systems. Each list is agreed with its own data owner. In course 1's file `NONE` is a real answer; in course 2's world it means no value.

🧪 **Your turn #3.** That one SQL line does the jobs of **two** of your functions. Which two? Give their names as a list of strings.

In [ ]:
answer_3 = ___      # a list of two function names, like ["match_key", "..."]
check.p5_ex3(answer_3)

## Wrap-up: what you can do now
- **Part 0:** a function is a recipe of named steps, and you can run them one at a time to see what each does.
- **Part 1:** strings, methods, `==`, f-strings and your first `def`.
- **Part 2:** lists, dictionaries, `for` and `if`, and `None`.
- **Part 3:** pandas: a question of every value, `.str`, `.loc`, `.mask`.
- **Part 4:** patterns for IDs and dates.
- **Part 5:** your own pipeline, proved against course 1, and the same steps read in SQL.

Most important of all: **check your code against something you trust.** Every exercise here compared your work with course 1's, cell by cell. Do the same with anything you build.

**Next: course 2,** where these steps run every day in a lakehouse, written in SQL.

## Answers

**#1**
```python
cleaned = [clean_value(v, codes) for v in out[column]]
```
**#2**
```python
df = my_strip_and_null(df)
```
**#3**
```python
answer_3 = ["my_strip_and_null", "clean_value"]
```